In [ ]:
from pathlib import Path

import joblib
import streamlit as st

from py import data_source, sql
from py.expected_points import (
    FEATURE_COLUMNS,
    build_pre_shot_features,
    train_expected_points_model,
)

sql_lite_connect = st.secrets["nda_gbb_connection"]["DB_CONNECTION"]
shots = data_source.run_query(
    sql=sql.get_play_by_play_sql(),
    connection=sql_lite_connect,
)
featured_shots = build_pre_shot_features(shots)
print(
    f"Loaded {len(featured_shots):,} shots across "
    f"{featured_shots['GAME_ID'].nunique():,} games."
)
print(f"Shared pre-shot features: {FEATURE_COLUMNS}")


In [ ]:
training_result = train_expected_points_model(
    shots,
    n_splits=3,
    holdout_fraction=0.2,
)

print(f"Selected model: {training_result.selected_model_name}")
display(training_result.validation_scores)
print("Future-game holdout metrics:")
print(training_result.holdout_metrics)


In [ ]:
project_root = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "py" / "expected_points.py").is_file()
    ),
    None,
)
if project_root is None:
    raise FileNotFoundError(
        "Could not locate the NDA_GBB_Stats project root from the notebook cwd."
    )

pipeline_path = project_root / "pipeline.pkl"
joblib.dump(training_result.model, pipeline_path)
print(f"Saved trained model to {pipeline_path}")
